In [2]:
"""
Modelo GRU para previsão de visualizações diárias do Portal do TRT18.
Mesma metodologia dos demais modelos: divisão sequencial 80/20, features
de calendário (mesmas do SARIMAX/Prophet) e defasagens da variável-alvo
(mesmas do SVR). Métricas: MAE, RMSE e R².
"""

import sys
print(sys.executable)
 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
 
# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
# -----------------------------------------------------------------------------
# 1) Leitura da base tratada comum
# -----------------------------------------------------------------------------
df = pd.read_csv("../dados/trafego_tratado.csv",
                 parse_dates=["Data"])
df = df.sort_values("Data").reset_index(drop=True)
 
# -----------------------------------------------------------------------------
# 2) Engenharia de atributos (mesma lógica do SVR)
#    - defasagens de 1, 7 e 30 dias
#    - médias móveis de 7 e 30 dias com shift(1) (sem informação contemporânea)
# -----------------------------------------------------------------------------
df["lag_1"] = df["Visualizações"].shift(1)
df["lag_7"] = df["Visualizações"].shift(7)
df["lag_30"] = df["Visualizações"].shift(30)
df["mm_7"] = df["Visualizações"].shift(1).rolling(7).mean()
df["mm_30"] = df["Visualizações"].shift(1).rolling(30).mean()
 
# Dummies de dia da semana (mesmas do SARIMAX, evita usar fim_de_semana
# que é combinação linear das dummies)
df = pd.get_dummies(df, columns=["dia_semana"], prefix="dow", drop_first=True)
 
# Features finais — mesmo conjunto exógeno dos demais modelos
features_calendario = [
    "recesso_judiciario", "feriado_nacional_fixo", "carnaval",
    "quarta_cinzas", "sexta_paixao", "corpus_christi",
    "data_especifica_judiciario", "ponto_facultativo_emenda",
]
features_dow = [c for c in df.columns if c.startswith("dow_")]
features_lag = ["lag_1", "lag_7", "lag_30", "mm_7", "mm_30"]
 
features = features_calendario + features_dow + features_lag
target = "Visualizações"
 
# Remove linhas iniciais sem histórico suficiente (por causa das defasagens)
df = df.dropna(subset=features + [target]).reset_index(drop=True)
 
# Garante tipos numéricos
df[features] = df[features].astype(float)
 
# -----------------------------------------------------------------------------
# 3) Divisão treino/teste sequencial — 80/20, mesma regra dos demais modelos
# -----------------------------------------------------------------------------
n = len(df)
n_treino = int(n * 0.8)
 
X = df[features].values
y = df[target].values.reshape(-1, 1)
datas = df["Data"].values
 
X_treino_raw, X_teste_raw = X[:n_treino], X[n_treino:]
y_treino_raw, y_teste_raw = y[:n_treino], y[n_treino:]
datas_teste = datas[n_treino:]
 
# -----------------------------------------------------------------------------
# 4) Normalização — ajustada APENAS no treino, igual ao SVR
# -----------------------------------------------------------------------------
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
 
X_treino_n = scaler_X.fit_transform(X_treino_raw)
X_teste_n = scaler_X.transform(X_teste_raw)
y_treino_n = scaler_y.fit_transform(y_treino_raw)
y_teste_n = scaler_y.transform(y_teste_raw)
 
# -----------------------------------------------------------------------------
# 5) Construção das sequências para a GRU
#    Janela de 14 dias: o modelo recebe os 14 dias anteriores (features) para
#    prever o dia seguinte.
# -----------------------------------------------------------------------------
JANELA = 14
 
def montar_sequencias(X_arr, y_arr, janela):
    Xs, ys = [], []
    for i in range(janela, len(X_arr)):
        Xs.append(X_arr[i - janela:i])
        ys.append(y_arr[i])
    return np.array(Xs), np.array(ys)
 
X_tr, y_tr = montar_sequencias(X_treino_n, y_treino_n, JANELA)
 
# Para o teste, concatena o final do treino para preservar a janela inicial
X_full_n = np.vstack([X_treino_n, X_teste_n])
y_full_n = np.vstack([y_treino_n, y_teste_n])
X_te, y_te = montar_sequencias(
    X_full_n[n_treino - JANELA:],
    y_full_n[n_treino - JANELA:],
    JANELA,
)
datas_te = datas_teste  # mesmo tamanho de y_te
 
# Tensores
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
X_te_t = torch.tensor(X_te, dtype=torch.float32)
y_te_t = torch.tensor(y_te, dtype=torch.float32)
 
# -----------------------------------------------------------------------------
# 6) Arquitetura GRU
# -----------------------------------------------------------------------------
class ModeloGRU(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]  # último passo
        out = self.dropout(out)
        return self.fc(out)
 
 
modelo = ModeloGRU(n_features=X_tr.shape[2], hidden_size=64, num_layers=1, dropout=0.1)
otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
criterio = nn.MSELoss()
 
# -----------------------------------------------------------------------------
# 7) Treinamento (mini-batches, com early stop simples por paciência)
# -----------------------------------------------------------------------------
EPOCAS = 200
BATCH = 32
PACIENCIA = 20
 
melhor_loss = float("inf")
contador = 0
melhor_estado = None
 
n_amostras = X_tr_t.shape[0]
 
for epoca in range(EPOCAS):
    modelo.train()
    perm = torch.randperm(n_amostras)
    perdas = []
    for i in range(0, n_amostras, BATCH):
        idx = perm[i:i + BATCH]
        xb, yb = X_tr_t[idx], y_tr_t[idx]
        otimizador.zero_grad()
        pred = modelo(xb)
        loss = criterio(pred, yb)
        loss.backward()
        otimizador.step()
        perdas.append(loss.item())
 
    perda_epoca = float(np.mean(perdas))
    if perda_epoca < melhor_loss - 1e-5:
        melhor_loss = perda_epoca
        contador = 0
        melhor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
    else:
        contador += 1
        if contador >= PACIENCIA:
            break
 
if melhor_estado is not None:
    modelo.load_state_dict(melhor_estado)
 
# -----------------------------------------------------------------------------
# 8) Avaliação no conjunto de teste
# -----------------------------------------------------------------------------
modelo.eval()
with torch.no_grad():
    pred_te_n = modelo(X_te_t).numpy()
 
pred_te = scaler_y.inverse_transform(pred_te_n).ravel()
y_te_real = scaler_y.inverse_transform(y_te_t.numpy()).ravel()
 
mae = mean_absolute_error(y_te_real, pred_te)
rmse = np.sqrt(mean_squared_error(y_te_real, pred_te))
r2 = r2_score(y_te_real, pred_te)
 
print(f"GRU - MAE:  {mae:,.2f}")
print(f"GRU - RMSE: {rmse:,.2f}")
print(f"GRU - R²:   {r2:.4f}")
 
# -----------------------------------------------------------------------------
# 9) Gráfico individual: valores reais vs previstos (GRU)
# -----------------------------------------------------------------------------
os.makedirs("../resultados", exist_ok=True) if False else os.makedirs("resultados", exist_ok=True)
 
plt.figure(figsize=(12, 5))
plt.plot(datas_te, y_te_real, label="Real", linewidth=1.2)
plt.plot(datas_te, pred_te, label="Previsto (GRU)", linewidth=1.2)
plt.title("GRU — Visualizações diárias (real vs. previsto)")
plt.xlabel("Data")
plt.ylabel("Visualizações")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("resultados/gru_real_vs_previsto.png", dpi=120)
plt.close()
 
# -----------------------------------------------------------------------------
# 10) Salva previsões para o gráfico de comparação final
# -----------------------------------------------------------------------------
pd.DataFrame({
    "Data": datas_te,
    "Real": y_te_real,
    "GRU": pred_te,
}).to_csv("resultados/previsoes_gru.csv", index=False)
 
print("\nArquivos gerados:")
print(" - resultados/gru_real_vs_previsto.png")
print(" - resultados/previsoes_gru.csv")
 

c:\Users\DIOGO\Desktop\Vs Code Git Hub\mp-Diogo\.venv\Scripts\python.exe
GRU - MAE:  4,306.43
GRU - RMSE: 6,811.94
GRU - R²:   0.8353

Arquivos gerados:
 - resultados/gru_real_vs_previsto.png
 - resultados/previsoes_gru.csv
